In [1]:
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

In [3]:
price_model = joblib.load("../models/catboost_price_model.pkl")

production_model = joblib.load("../models/catboost_production.pkl")

In [5]:
price = pd.read_csv("../datasets/modeling/price_features.csv")

production = pd.read_csv("../datasets/processed/production_features.csv")

In [6]:
price_features = price_model.feature_names_
print(price_features)

['State', 'Year', 'Month', 'Standard_Crop', 'Market_Records', 'Unique_Markets', 'Avg_Temperature_C', 'Avg_DewPoint_C', 'Avg_Rainfall_mm', 'Avg_Wind_Speed', 'Avg_Surface_Pressure_hPa', 'Grid_Points', 'Price_Lag_1', 'Price_Lag_2', 'Price_Lag_3', 'Rolling_Mean_3', 'Rolling_Mean_6', 'Rolling_STD_3', 'Quarter', 'Month_Sin', 'Month_Cos', 'Previous_Change', 'Previous_Growth', 'Temp_Rainfall', 'Temp_DewPoint', 'Rainfall_Wind']


Get Historical Data

In [11]:
def get_price_history(state, crop):

    history = (
        price[
            (price["State"] == state) &
            (price["Standard_Crop"] == crop)
        ]
        .sort_values(["Year", "Month"])
        .reset_index(drop=True)
        .copy()
    )

    if history.empty:
        raise ValueError(
            f"No historical data found for {crop} in {state}"
        )

    return history

Testing 

In [12]:
history = get_price_history(
    state="Andhra Pradesh",
    crop="Paddy"
)

history.tail()

,State,Year,Month,Standard_Crop,Avg_Modal_Price,Median_Modal_Price,Price_STD,Min_Modal_Price,Max_Modal_Price,Market_Records,...,Rolling_Mean_6,Rolling_STD_3,Quarter,Month_Sin,Month_Cos,Previous_Change,Previous_Growth,Temp_Rainfall,Temp_DewPoint,Rainfall_Wind
140,Andhra Pradesh,2015,8,Paddy,1504.249665,1390.0,250.158328,1050.0,4900.0,4478,...,7.296011,0.006843,3,-8.660254e-01,-5.000000e-01,0.002324,0.000318,141.373320,5.582315,13.658110
141,Andhra Pradesh,2015,9,Paddy,1544.259930,1400.0,310.231207,1050.0,6300.0,4305,...,7.300125,0.008636,3,-1.000000e+00,-1.836970e-16,0.013661,0.001871,173.622400,4.956652,9.474909
142,Andhra Pradesh,2015,10,Paddy,1570.087388,1410.0,314.429036,1050.0,5900.0,4337,...,7.307654,0.020275,4,-8.660254e-01,5.000000e-01,0.026233,0.003585,77.237680,5.666407,1.891325
143,Andhra Pradesh,2015,11,Paddy,1585.047422,1430.0,322.745410,1050.0,3500.0,4112,...,7.318863,0.021585,4,-5.000000e-01,8.660254e-01,0.016576,0.002257,180.031130,4.050363,14.708597
144,Andhra Pradesh,2015,12,Paddy,1597.714948,1430.0,457.412055,1110.0,13900.0,4315,...,7.331995,0.013187,4,-2.449294e-16,1.000000e+00,0.009477,0.001288,16.463633,5.091457,0.938856


Prepare Price Features

In [13]:
def prepare_price_features(
    history,
    year,
    month,
    temperature,
    dew_point,
    rainfall,
    wind_speed,
    pressure
):

    last = history.iloc[-1]

    features = {
        "State": last["State"],
        "Year": year,
        "Month": month,
        "Standard_Crop": last["Standard_Crop"],

        "Market_Records": last["Market_Records"],
        "Unique_Markets": last["Unique_Markets"],

        "Avg_Temperature_C": temperature,
        "Avg_DewPoint_C": dew_point,
        "Avg_Rainfall_mm": rainfall,
        "Avg_Wind_Speed": wind_speed,
        "Avg_Surface_Pressure_hPa": pressure,

        "Grid_Points": last["Grid_Points"],

        "Price_Lag_1": last["Price_Lag_1"],
        "Price_Lag_2": last["Price_Lag_2"],
        "Price_Lag_3": last["Price_Lag_3"],

        "Rolling_Mean_3": last["Rolling_Mean_3"],
        "Rolling_Mean_6": last["Rolling_Mean_6"],
        "Rolling_STD_3": last["Rolling_STD_3"],

        "Quarter": (month - 1) // 3 + 1,

        "Month_Sin": np.sin(2 * np.pi * month / 12),
        "Month_Cos": np.cos(2 * np.pi * month / 12),

        "Previous_Change": last["Previous_Change"],
        "Previous_Growth": last["Previous_Growth"],

        "Temp_Rainfall": temperature * rainfall,
        "Temp_DewPoint": temperature * dew_point,
        "Rainfall_Wind": rainfall * wind_speed,
    }

    features = pd.DataFrame([features])

    return features[price_model.feature_names_]

Testing

In [14]:
sample = prepare_price_features(
    history=history,
    year=2016,
    month=1,
    temperature=history.iloc[-1]["Avg_Temperature_C"],
    dew_point=history.iloc[-1]["Avg_DewPoint_C"],
    rainfall=history.iloc[-1]["Avg_Rainfall_mm"],
    wind_speed=history.iloc[-1]["Avg_Wind_Speed"],
    pressure=history.iloc[-1]["Avg_Surface_Pressure_hPa"]
)

sample

,State,Year,Month,Standard_Crop,Market_Records,Unique_Markets,Avg_Temperature_C,Avg_DewPoint_C,Avg_Rainfall_mm,Avg_Wind_Speed,...,Rolling_Mean_6,Rolling_STD_3,Quarter,Month_Sin,Month_Cos,Previous_Change,Previous_Growth,Temp_Rainfall,Temp_DewPoint,Rainfall_Wind
0,Andhra Pradesh,2016,1,Paddy,4315,131,24.468927,19.37747,0.672838,1.395366,...,7.331995,0.013187,1,0.5,0.866025,0.009477,0.001288,16.463632,474.145899,0.938856


In [15]:
def predict_next_price(features):

    log_price = price_model.predict(features)[0]

    actual_price = np.expm1(log_price)

    return log_price, actual_price
    

In [16]:
log_price, predicted_price = predict_next_price(sample)

print(f"Log Price : {log_price:.4f}")
print(f"Price     : ₹{predicted_price:.2f}")

Log Price : 7.3227
Price     : ₹1513.30


In [17]:
def update_price_history(
    history,
    predicted_log_price,
    year,
    month,
    temperature,
    dew_point,
    rainfall,
    wind_speed,
    pressure
):

    last = history.iloc[-1].copy()

    new_row = last.copy()

    # -----------------------------
    # Update time
    # -----------------------------
    new_row["Year"] = year
    new_row["Month"] = month
    new_row["Quarter"] = (month - 1) // 3 + 1

    new_row["Month_Sin"] = np.sin(2 * np.pi * month / 12)
    new_row["Month_Cos"] = np.cos(2 * np.pi * month / 12)

    # -----------------------------
    # Predicted price
    # -----------------------------
    new_row["Log_Price"] = predicted_log_price
    new_row["Avg_Modal_Price"] = np.expm1(predicted_log_price)

    # -----------------------------
    # Shift lag features
    # -----------------------------
    new_row["Price_Lag_3"] = last["Price_Lag_2"]
    new_row["Price_Lag_2"] = last["Price_Lag_1"]
    new_row["Price_Lag_1"] = predicted_log_price

    # -----------------------------
    # Weather
    # -----------------------------
    new_row["Avg_Temperature_C"] = temperature
    new_row["Avg_DewPoint_C"] = dew_point
    new_row["Avg_Rainfall_mm"] = rainfall
    new_row["Avg_Wind_Speed"] = wind_speed
    new_row["Avg_Surface_Pressure_hPa"] = pressure

    # -----------------------------
    # Interaction features
    # -----------------------------
    new_row["Temp_Rainfall"] = temperature * rainfall
    new_row["Temp_DewPoint"] = temperature * dew_point
    new_row["Rainfall_Wind"] = rainfall * wind_speed

    # -----------------------------
    # Append row
    # -----------------------------
    history = pd.concat(
        [history, pd.DataFrame([new_row])],
        ignore_index=True
    )

    # -----------------------------
    # Recompute rolling features
    # -----------------------------
    logs = history["Log_Price"]

    history.loc[
        history.index[-1],
        "Rolling_Mean_3"
    ] = logs.iloc[-3:].mean()

    history.loc[
        history.index[-1],
        "Rolling_Mean_6"
    ] = logs.iloc[-6:].mean()

    history.loc[
        history.index[-1],
        "Rolling_STD_3"
    ] = logs.iloc[-3:].std()

    # -----------------------------
    # Previous Change
    # -----------------------------
    history.loc[
        history.index[-1],
        "Previous_Change"
    ] = (
        predicted_log_price -
        last["Log_Price"]
    )

    # -----------------------------
    # Previous Growth
    # -----------------------------
    history.loc[
        history.index[-1],
        "Previous_Growth"
    ] = (
        history.iloc[-1]["Previous_Change"] /
        last["Log_Price"]
    )

    return history

In [31]:
def forecast_prices(
    state,
    crop,
    start_year,
    start_month,
    months,
    weather_data
):

    history = get_price_history(state, crop)

    forecasts = []

    year = start_year
    month = start_month

    for i in range(months):

        weather = weather_data.iloc[i]

        features = prepare_price_features(
            history=history,
            year=year,
            month=month,
            temperature=weather["Avg_Temperature_C"],
            dew_point=weather["Avg_DewPoint_C"],
            rainfall=weather["Avg_Rainfall_mm"],
            wind_speed=weather["Avg_Wind_Speed"],
            pressure=weather["Avg_Surface_Pressure_hPa"]
        )

        log_price, predicted_price = predict_next_price(features)

        forecasts.append({
            "State": state,
            "Crop": crop,
            "Year": year,
            "Month": month,
            "Predicted_Log_Price": log_price,
            "Predicted_Price": predicted_price
        })

        history = update_price_history(
            history,
            predicted_log_price=log_price,
            year=year,
            month=month,
            temperature=weather["Avg_Temperature_C"],
            dew_point=weather["Avg_DewPoint_C"],
            rainfall=weather["Avg_Rainfall_mm"],
            wind_speed=weather["Avg_Wind_Speed"],
            pressure=weather["Avg_Surface_Pressure_hPa"]
        )

        month += 1

        if month > 12:
            month = 1
            year += 1

    return pd.DataFrame(forecasts), history

In [32]:
last_weather = history.iloc[-1]

weather_data = pd.DataFrame([
    {
        "Avg_Temperature_C": last_weather["Avg_Temperature_C"],
        "Avg_DewPoint_C": last_weather["Avg_DewPoint_C"],
        "Avg_Rainfall_mm": last_weather["Avg_Rainfall_mm"],
        "Avg_Wind_Speed": last_weather["Avg_Wind_Speed"],
        "Avg_Surface_Pressure_hPa": last_weather["Avg_Surface_Pressure_hPa"]
    }
] * 6)

In [33]:
forecast = forecast_prices(
    state="Andhra Pradesh",
    crop="Paddy",
    start_year=2016,
    start_month=1,
    months=6,
    weather_data=weather_data
)

forecast

(            State   Crop  Year  Month  Predicted_Log_Price  Predicted_Price
 0  Andhra Pradesh  Paddy  2016      1             7.322708      1513.299033
 1  Andhra Pradesh  Paddy  2016      2             7.255850      1415.366872
 2  Andhra Pradesh  Paddy  2016      3             7.222184      1368.476426
 3  Andhra Pradesh  Paddy  2016      4             7.198123      1335.918581
 4  Andhra Pradesh  Paddy  2016      5             7.196298      1333.481626
 5  Andhra Pradesh  Paddy  2016      6             7.203510      1343.140079,
               State  Year  Month Standard_Crop  Avg_Modal_Price  \
 0    Andhra Pradesh  2003     12         Paddy       693.333333   
 1    Andhra Pradesh  2004      1         Paddy       667.247475   
 2    Andhra Pradesh  2004      2         Paddy       600.447917   
 3    Andhra Pradesh  2004      3         Paddy       607.333333   
 4    Andhra Pradesh  2004      4         Paddy       614.514184   
 ..              ...   ...    ...           ...     

In [34]:
forecast_prices(
    state="Punjab",
    crop="Wheat",
    start_year=2016,
    start_month=1,
    months=6,
    weather_data=weather_data
)

(    State   Crop  Year  Month  Predicted_Log_Price  Predicted_Price
 0  Punjab  Wheat  2016      1             7.294997      1471.912148
 1  Punjab  Wheat  2016      2             7.297927      1476.234645
 2  Punjab  Wheat  2016      3             7.301550      1481.595557
 3  Punjab  Wheat  2016      4             7.275589      1443.601652
 4  Punjab  Wheat  2016      5             7.235955      1387.466298
 5  Punjab  Wheat  2016      6             7.223377      1370.111855,
       State  Year  Month Standard_Crop  Avg_Modal_Price  Median_Modal_Price  \
 0    Punjab  2002      8         Wheat       548.964912               550.0   
 1    Punjab  2002      9         Wheat       566.780488               565.0   
 2    Punjab  2002     10         Wheat       566.666667               589.0   
 3    Punjab  2002     11         Wheat       575.807692               571.5   
 4    Punjab  2002     12         Wheat       587.557377               580.0   
 ..      ...   ...    ...           

In [35]:
forecast_prices(
    state="Karnataka",
    crop="Maize",
    start_year=2016,
    start_month=6,
    months=6,
    weather_data=weather_data
)

(       State   Crop  Year  Month  Predicted_Log_Price  Predicted_Price
 0  Karnataka  Maize  2016      6             7.184683      1318.070996
 1  Karnataka  Maize  2016      7             7.172410      1301.981474
 2  Karnataka  Maize  2016      8             7.174075      1304.151692
 3  Karnataka  Maize  2016      9             7.161256      1287.528805
 4  Karnataka  Maize  2016     10             7.201566      1340.529912
 5  Karnataka  Maize  2016     11             7.252862      1411.140826,
          State  Year  Month Standard_Crop  Avg_Modal_Price  \
 0    Karnataka  2002      1         Maize       459.088235   
 1    Karnataka  2002      2         Maize       489.131579   
 2    Karnataka  2002      3         Maize       513.312500   
 3    Karnataka  2002      4         Maize       516.861004   
 4    Karnataka  2002      5         Maize       500.567474   
 ..         ...   ...    ...           ...              ...   
 169  Karnataka  2016      7         Maize      1301.9